In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

import os

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [4]:
DATA_PATH = "../data/movie_dataset_clean.parquet"

df = pd.read_parquet(DATA_PATH)

df.head()

,id,title,overview,genres,cast,directors,keywords,release_date,vote_average,vote_count,runtime,year,document
0,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...","Andrew Stanton, Lee Unkrich","sydney, australia, parent child relationship, ...",2003-05-30 00:00:00,7.824,18061,100,2003.0,"Title: Finding Nemo\nOverview: Nemo, an advent..."
1,14,American Beauty,"Lester Burnham, a depressed suburban father in...",Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"estate agent, adultery, coming out, first time...",1999-09-15 00:00:00,8.0,11260,122,1999.0,Title: American Beauty\nOverview: Lester Burnh...
2,16,Dancer in the Dark,"Selma, a Czech immigrant on the verge of blind...","Drama, Crime","Björk, Catherine Deneuve, David Morse, Peter S...",Lars von Trier,"factory worker, dying and death, individual, i...",2000-06-30 00:00:00,7.869,1618,141,2000.0,"Title: Dancer in the Dark\nOverview: Selma, a ..."
3,17,The Dark,"In an attempt to pull her family together, Adè...","Horror, Thriller, Mystery","Maria Bello, Sean Bean, Abigail Stone, Richard...",John Fawcett,"sea, wales, child abuse, shepherd, adolescence...",2005-09-28 00:00:00,5.755,247,87,2005.0,Title: The Dark\nOverview: In an attempt to pu...
4,20,My Life Without Me,A fatally ill mother with only two months to l...,"Drama, Romance","Sarah Polley, Amanda Plummer, Scott Speedman, ...",Isabel Coixet,"dying and death, daughter, farewell, night shi...",2003-03-07 00:00:00,5.941,421,106,2003.0,Title: My Life Without Me\nOverview: A fatally...


In [5]:
df.shape

(12338, 13)

In [6]:
df.columns

Index(['id', 'title', 'overview', 'genres', 'cast', 'directors', 'keywords',
       'release_date', 'vote_average', 'vote_count', 'runtime', 'year',
       'document'],
      dtype='object')

In [7]:
print(df.iloc[0]["document"])

Title: Finding Nemo
Overview: Nemo, an adventurous young clownfish, is unexpectedly taken from his Great Barrier Reef home to a dentist's office aquarium. It's up to his worrisome father Marlin and a friendly but forgetful fish Dory to bring Nemo home -- meeting vegetarian sharks, surfer dude turtles, hypnotic jellyfish, hungry seagulls, and more along the way.
Genres: Animation, Family
Director: Andrew Stanton, Lee Unkrich
Cast: Albert Brooks, Ellen DeGeneres, Alexander Gould, Willem Dafoe, Geoffrey Rush, Brad Garrett, Allison Janney, Austin Pendleton, Stephen Root, Vicki Lewis
Keywords: sydney, australia, parent child relationship, anthropomorphism, harbor, underwater, shark, pelican, fish tank, great barrier reef, sea turtle, missing child, aftercreditsstinger, duringcreditsstinger, short term memory loss, clownfish, father son reunion, protective father
Release Date: 2003-05-30 00:00:00
Rating: 7.824
Runtime: 100


In [8]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"


model = SentenceTransformer(
    MODEL_NAME
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
test_embedding = model.encode(
    "A science fiction movie about space"
)

test_embedding.shape

(384,)

In [10]:
documents = df["document"].tolist()

len(documents)

12338

In [11]:
embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/386 [00:00<?, ?it/s]

In [12]:
embeddings.shape

(12338, 384)

In [13]:
embeddings = np.array(
    embeddings
)

In [14]:
type(embeddings)

numpy.ndarray

In [15]:
EMBEDDING_PATH = "../data/movie_embeddings.npy"


np.save(
    EMBEDDING_PATH,
    embeddings
)

In [16]:
loaded_embeddings = np.load(
    EMBEDDING_PATH
)

In [17]:
loaded_embeddings.shape

(12338, 384)

In [19]:
#save metadata
metadata = df[
    [
        "id",
        "title",
        "document"
    ]
]

In [20]:
metadata.to_parquet(
    "../data/movie_metadata.parquet",
    index=False
)

In [21]:
#test with similarity test
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
query = "A movie about space exploration and astronauts"

In [23]:
query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

In [25]:
scores = cosine_similarity(
    [query_embedding],
    embeddings
)[0]

In [26]:
top_indices = scores.argsort()[-5:][::-1]

top_indices

array([11090, 12182,  4244,  6812,  1068])

In [27]:
df.iloc[
    top_indices
][
    [
        "title",
        "overview",
        "vote_average"
    ]
]

,title,overview,vote_average
11090,3022,A group of astronauts living in the haunting e...,5.402
12182,Return to Space,The inspirational rise of SpaceX as well as El...,6.514
4244,Gravity,"Dr. Ryan Stone, a brilliant medical engineer o...",7.162
6812,Passengers,A spacecraft traveling to a distant colony pla...,6.934
1068,Red Planet,Astronauts search for solutions to save a dyin...,5.691
